# Building the FOMC corpus files

**ECON5129 Statistical Machine Learning** &middot; Adam Smith Business School, University of Glasgow

This notebook is run **once** by the course convener, not by students. It downloads the ConvoKit distribution of Federal Open Market Committee meeting transcripts, reduces it to two compact files, and writes them into the `data` folder of the course repository. Labs 9 and 10 then read those files in under a second, with no large download and no extra dependency.

Run it in Google Colab, then download the two output files and commit them to the repository.

**Source and citation.** Transcripts of FOMC meetings from 1977 to 2008, distributed with:

> Tan, C. and L. Lee (2016). Talk it up or play it down? (Un)expected correlations between (de-)emphasis and recurrence of discussion points in consequential U.S. economic policy meetings. *Text As Data*.

The underlying transcripts are US government works. The ConvoKit packaging should be cited as above wherever the derived files are used.

In [ ]:
# !pip install convokit --quiet

In [ ]:
import numpy as np
import pandas as pd
from convokit import Corpus, download

corpus = Corpus(filename=download("fomc-corpus"))
corpus.print_summary_stats()

## Meeting-level file

Each conversation is one meeting, indexed by its date. Utterances are concatenated in speech order to give one document per meeting. The spaCy parses stored in the utterance metadata are discarded, since they account for most of the size of the corpus and none of its use here.

In [ ]:
def speaker_is_chair(speaker):
    """Read the chair flag from speaker metadata, tolerating missing fields."""
    meta = getattr(speaker, "meta", {}) or {}
    return bool(meta.get("chair", False))


records = []
for conversation in corpus.iter_conversations():
    utterances = sorted(
        conversation.iter_utterances(),
        key=lambda u: (u.meta or {}).get("speech_index", 0),
    )
    texts = [u.text for u in utterances if isinstance(u.text, str) and u.text.strip()]
    speakers = {u.speaker.id for u in utterances}
    chairs = sorted({u.speaker.id for u in utterances if speaker_is_chair(u.speaker)})

    document = " ".join(texts)
    records.append({
        "date": pd.to_datetime(conversation.id, errors="coerce"),
        "chair": chairs[0] if chairs else "",
        "n_speakers": len(speakers),
        "n_utterances": len(texts),
        "n_words": len(document.split()),
        "text": document,
    })

meetings = pd.DataFrame(records).dropna(subset=["date"]).sort_values("date")
meetings["year"] = meetings["date"].dt.year
meetings = meetings[["date", "year", "chair", "n_speakers", "n_utterances", "n_words", "text"]]
meetings = meetings.reset_index(drop=True)

print(f"{len(meetings)} meetings from {meetings['date'].min():%Y-%m} to {meetings['date'].max():%Y-%m}")
print(meetings[["date", "chair", "n_speakers", "n_utterances", "n_words"]].head())

## Utterance-level file

A speaker-level file supports the take-home challenge on chair versus committee tone. The full set of utterances is far larger than the labs need, so a capped sample per meeting is stored.

In [ ]:
max_per_meeting = 80

utterance_records = []
for conversation in corpus.iter_conversations():
    date = pd.to_datetime(conversation.id, errors="coerce")
    if pd.isna(date):
        continue
    utterances = sorted(
        conversation.iter_utterances(),
        key=lambda u: (u.meta or {}).get("speech_index", 0),
    )
    for u in utterances[:max_per_meeting]:
        if not isinstance(u.text, str) or not u.text.strip():
            continue
        utterance_records.append({
            "date": date,
            "speaker": u.speaker.id,
            "chair": speaker_is_chair(u.speaker),
            "speech_index": (u.meta or {}).get("speech_index", 0),
            "text": u.text,
        })

utterances_frame = pd.DataFrame(utterance_records).sort_values(["date", "speech_index"])
print(f"{len(utterances_frame):,} utterances retained")

## Write the files

Both files are gzip compressed. `pandas` reads them directly from a URL, so the labs need no unpacking step.

In [ ]:
meetings.to_csv("fomc_meetings.csv.gz", index=False, compression="gzip")
utterances_frame.to_csv("fomc_utterances.csv.gz", index=False, compression="gzip")

import os

for name in ["fomc_meetings.csv.gz", "fomc_utterances.csv.gz"]:
    print(f"{name}: {os.path.getsize(name) / 1e6:.1f} MB")

## Validation

Confirm that the files load through the same code path the labs use before committing them.

In [ ]:
check = pd.read_csv("fomc_meetings.csv.gz")
check["date"] = pd.to_datetime(check["date"])

assert {"date", "year", "chair", "n_speakers", "n_utterances", "n_words", "text"} <= set(check.columns)
assert check["text"].str.len().min() > 0
assert check["date"].is_monotonic_increasing

print(f"meetings: {len(check)}")
print(f"median length: {check['n_words'].median():,.0f} words")
print(f"chairs present: {', '.join(sorted(c for c in check['chair'].unique() if c))}")

## Next steps

1. Download `fomc_meetings.csv.gz` and `fomc_utterances.csv.gz` from the Colab file browser.
2. Place both in the `data` folder of the repository.
3. Commit and push.

Labs 9 and 10 will then load them through `e5.load_fomc_meetings()` with no further configuration.